# APEC Project — Data Analysis

This notebook builds up, CTE by CTE, the two core queries that run in `personalization_impact_v2.ipynb`: Part 1's `apec_split_query` and Part 2's `category_query`. Each step below adds exactly one CTE and re-runs, so the final step in Part B and Part C reproduces the exact query and numbers used there.

For readability, every step here is scoped to a single illustrative month (April 2026) rather than the full year-plus window the real notebook uses — the query shape and every join are identical, just over a smaller slice of time so the intermediate row counts stay easy to look at.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)

MONTH_START = '2026-04-01'
MONTH_END = '2026-05-01'
DORMANT_3YR_MIN_DAYS = 1095
SESSION_ATTRIBUTION_WINDOW_DAYS = 7

## Part A — The shared pipeline

Both queries in `personalization_impact_v2.ipynb` are built from the same four CTEs, just grouped differently at the end. Part A builds those four up one at a time.

### A1 — the `sends` CTE alone

Every Blueshift send in the month, no filtering by client or campaign type yet — just the date window and `holdout_group = 0` (excluding suppressed/holdout sends, which aren't part of the addressable campaign population).

In [2]:
query(f"""--sql
SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
FROM blueshift.campaign_activity_kpis
WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
  AND holdout_group = 0
ORDER BY client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,send_utm_campaign,send_utm_content
0,3,2026-04-10 16:28:45.062,email_us_w_dse_freestyle_centralized_aprildse_041026_active,email_us_w_dse_freestyle_centralized_aprildse_041026_act...
1,3,2026-04-30 14:26:34.608,email_us_w_static_freestyle_q4extrapercentoffclearance_l...,email_us_w_static_freestyle_q4extrapercentoffclearance_l...
2,3,2026-04-13 14:50:08.446,email_us_w_dse_freestyle_centralized_aprildse_041326_active,email_us_w_dse_freestyle_centralized_aprildse_041326_act...
3,3,2026-04-25 14:14:02.341,email_us_w_dse_freestyle_centralized_aprildse_042526_active,email_us_w_dse_freestyle_centralized_aprildse_042526_act...
4,3,2026-04-02 14:25:13.059,email_us_w_static_freestyle_freshtakeonspringdenim_04022...,email_us_w_static_freestyle_freshtakeonspringdenim_04022...


In [3]:
query(f"""--sql
SELECT count(*) as n_sends, count(distinct client_id) as n_clients
FROM blueshift.campaign_activity_kpis
WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
  AND holdout_group = 0
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_sends,n_clients
0,249638382,7138407


249,638,382 sends across 7,138,407 distinct clients in April 2026 — this is StitchFix's entire send volume for the month, not yet restricted to the Dormant 3+ yrs population.

### A2 — add `dormant_3yr_sends`: the population filter

Join `sends` to `curated.checkout_based_client_state_journal` on `client_id`, with the send's own `sent_timestamp` falling inside a state period (`start_timestamp <= sent_timestamp < end_timestamp`) — this checks the client's lifecycle state **as of that specific send**, not their state today. Keep only `client_state_detail = 'Dormant'` with more than 1,095 days since their last checkout.

In [4]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
)
SELECT
    s.client_id, s.sent_timestamp, s.send_utm_campaign,
    j.client_state_detail, j.last_buyable_checkout_ts,
    date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) as days_since_checkout
FROM sends s
INNER JOIN curated.checkout_based_client_state_journal j
    ON s.client_id = j.client_id
    AND s.sent_timestamp >= j.start_timestamp
    AND s.sent_timestamp <  j.end_timestamp
WHERE j.client_state_detail = 'Dormant'
  AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
ORDER BY s.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,send_utm_campaign,client_state_detail,last_buyable_checkout_ts,days_since_checkout
0,63,2026-04-18 00:37:23.008,email_us_n_transactional_trackeditem_delivered,Dormant,2018-02-26 22:03:09.138,2973
1,63,2026-04-20 15:53:04.353,crumbs_has_app,Dormant,2018-02-26 22:03:09.138,2975
2,63,2026-04-15 18:06:48.672,email_n_transactional_fixshipped,Dormant,2018-02-26 22:03:09.138,2970
3,63,2026-04-11 14:00:36.071,email_n_transactional_fixpreview_review_v2,Dormant,2018-02-26 22:03:09.138,2966
4,63,2026-04-17 20:37:23.695,email_us_n_transactional_trackeditem_outfordelivery,Dormant,2018-02-26 22:03:09.138,2972


In [5]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
)
SELECT count(*) as n_sends, count(distinct s.client_id) as n_clients
FROM sends s
INNER JOIN curated.checkout_based_client_state_journal j
    ON s.client_id = j.client_id
    AND s.sent_timestamp >= j.start_timestamp
    AND s.sent_timestamp <  j.end_timestamp
WHERE j.client_state_detail = 'Dormant'
  AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_sends,n_clients
0,38144005,1198458


38,144,005 sends across 1,198,458 clients — down from 7.1M clients to ~1.2M once restricted to the Dormant 3+ yrs population.

### A3 — add `matched_sessions`: the session/UTM match

Join to `curated.user_session_conversion_metrics` on `client_id`, `utm_source = 'blueshift'`, and `utm_content` matching the send's own `send_utm_content` exactly, with the session landing within `SESSION_ATTRIBUTION_WINDOW_DAYS` of the send. This is the step that makes the metric **click-through-based**: a send only survives into this CTE if there's a real session carrying its UTMs.

In [6]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
)
SELECT d.client_id, d.sent_timestamp, u.active_session_id, u.datetime_in_utc, u.utm_content
FROM dormant_3yr_sends d
INNER JOIN curated.user_session_conversion_metrics u
    ON u.client_id = d.client_id
    AND u.utm_source = 'blueshift'
    AND u.utm_content = d.send_utm_content
    AND u.datetime_in_utc >= d.sent_timestamp
    AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
WHERE u.date_in_utc >= DATE '{MONTH_START}'
ORDER BY d.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,active_session_id,datetime_in_utc,utm_content
0,63,2026-04-12 02:00:38.788,f5e286ae-5dfe-423a-9208-e02481365def,2026-04-12 02:03:00.853,email_us_n_transactional_fixpreviewreminder_121136711138...
1,63,2026-04-21 18:37:49.217,cad002ab-1437-4ac8-bdf6-b747de30165b,2026-04-21 19:01:06.183,email_us_n_transactional_finalreturnreminder_12113720915...
2,2011004,2026-04-25 01:49:46.574,0d0b4fd2-dfe7-45c6-be0f-28a281b0bd67,2026-04-25 02:26:56.979,email_us_n_transactional_fixpreviewreminder_121136711138...
3,3003512,2026-04-30 17:02:23.459,9FC1A05C-C165-4F41-A207-FB0E232C2417,2026-04-30 17:02:30.096,push_us_w_freestyle_newarrivals_043026
4,3005408,2026-04-30 23:10:38.868,f56123fe-6bf1-4c16-8b07-c9c17c921bb1,2026-04-30 23:44:35.216,email_us_n_transactional_fix_scheduled_1205954616645804


In [7]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
)
SELECT count(*) as n_matched, count(distinct d.client_id) as n_clients
FROM dormant_3yr_sends d
INNER JOIN curated.user_session_conversion_metrics u
    ON u.client_id = d.client_id
    AND u.utm_source = 'blueshift'
    AND u.utm_content = d.send_utm_content
    AND u.datetime_in_utc >= d.sent_timestamp
    AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
WHERE u.date_in_utc >= DATE '{MONTH_START}'
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_matched,n_clients
0,53237,26938


26,938 of the 1,198,458 eligible clients (~2.2%) have any UTM-matched session at all this month — most sends never produce a trackable click-through, which is exactly why this metric reads as a lower bound on true reactivation.

### A4 — add `attributed_demand`: the demand confirmation

Join the matched session's `active_session_id` to `curated.client_reactivation_demand_events`, keeping only `demand_type IN ('fix', 'direct_buy')` — this confirms the session didn't just happen, it actually produced a real demand event.

In [8]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
matched_sessions AS (
    SELECT d.client_id, d.sent_timestamp, u.active_session_id
    FROM dormant_3yr_sends d
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = d.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = d.send_utm_content
        AND u.datetime_in_utc >= d.sent_timestamp
        AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
)
SELECT DISTINCT ms.client_id, ms.sent_timestamp, e.demand_id, e.demand_type, e.created_ts
FROM matched_sessions ms
INNER JOIN curated.client_reactivation_demand_events e
    ON e.active_session_id = ms.active_session_id
WHERE e.demand_type IN ('fix', 'direct_buy')
ORDER BY ms.client_id
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sent_timestamp,demand_id,demand_type,created_ts
0,3025345,2026-04-27 12:22:58.401,667900122,fix,2026-04-27 12:24:28.000
1,3025345,2026-04-27 12:22:58.401,667900118,fix,2026-04-27 12:24:21.031
2,3025345,2026-04-27 12:22:58.401,667900124,fix,2026-04-27 12:24:28.000
3,3025345,2026-04-27 12:22:58.401,667900125,fix,2026-04-27 12:24:28.000
4,3025345,2026-04-27 12:22:58.401,667900123,fix,2026-04-27 12:24:28.000


In [9]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
matched_sessions AS (
    SELECT d.client_id, d.sent_timestamp, u.active_session_id
    FROM dormant_3yr_sends d
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = d.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = d.send_utm_content
        AND u.datetime_in_utc >= d.sent_timestamp
        AND u.datetime_in_utc <  d.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
),
attributed_demand AS (
    SELECT DISTINCT ms.client_id, ms.sent_timestamp
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
)
SELECT count(*) as n_attributed, count(distinct client_id) as n_clients FROM attributed_demand
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_attributed,n_clients
0,3575,3505


3,505 clients confirmed reactivated this month — out of 26,938 with a matched session, out of 1,198,458 eligible. This is the numerator for the reactivation rate in both Part B and Part C.

## Part B — `apec_split_query` (Part 1 of `personalization_impact_v2.ipynb`)

Reuses the same `sends` → `dormant_3yr_sends` → `matched_sessions` → `attributed_demand` mechanics from Part A. Two things are added: a binary `apec_group` category on `dormant_3yr_sends`, and a `client_group` collapse that determines, per client per group, whether they reactivated at all.

### B1 — the `categorized_sends` CTE alone (binary)

Any campaign name containing "apec" is `APEC`; everything else is `non-APEC`. Transactional/account-service email (any campaign name containing "transactional" — password resets, order/Fix status confirmations, and similar) is excluded from the population entirely before this split, since a client only receives one of these because they're already taking an action on their own account, not because of a marketing choice.

In [10]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
      AND send_utm_campaign NOT LIKE '%transactional%'
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_campaign
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
)
SELECT
    CASE WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC' ELSE 'non-APEC' END AS apec_group,
    count(*) as n_sends, count(distinct client_id) as n_clients
FROM dormant_3yr_sends
GROUP BY 1
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,apec_group,n_sends,n_clients
0,APEC,1534243,203136
1,non-APEC,36345470,1161774


APEC: 203,136 clients | non-APEC: 1,161,774 clients — APEC-labeled sends are a small minority of this population's total send volume (transactional email already excluded, so this non-APEC count is smaller than the raw "everything but APEC" total would be).

### B2 — the full `apec_split_query`

Add `matched_sessions` and `attributed_demand` from Part A, then `client_group` (one row per client per `apec_group`, `MAX()` of whether they reactivated), then the final aggregation. This is the exact query behind Part 1's headline comparison.

In [11]:
apec_split_query = f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
      AND send_utm_campaign NOT LIKE '%transactional%'
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_campaign, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
categorized_sends AS (
    SELECT
        client_id, sent_timestamp, send_utm_content,
        CASE WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC' ELSE 'non-APEC' END AS apec_group
    FROM dormant_3yr_sends
),
matched_sessions AS (
    SELECT c.client_id, c.sent_timestamp, c.apec_group, u.active_session_id
    FROM categorized_sends c
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = c.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = c.send_utm_content
        AND u.datetime_in_utc >= c.sent_timestamp
        AND u.datetime_in_utc <  c.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
),
attributed_demand AS (
    SELECT DISTINCT ms.client_id, ms.sent_timestamp, ms.apec_group
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
),
client_group AS (
    SELECT
        d.client_id, d.apec_group,
        MAX(CASE WHEN a.client_id IS NOT NULL THEN 1 ELSE 0 END) AS client_reactivated
    FROM categorized_sends d
    LEFT JOIN attributed_demand a
        ON d.client_id = a.client_id AND d.sent_timestamp = a.sent_timestamp AND d.apec_group = a.apec_group
    GROUP BY 1, 2
)
SELECT
    apec_group,
    COUNT(DISTINCT client_id) AS unique_clients_sent,
    SUM(client_reactivated) AS reactivated_clients,
    CAST(SUM(client_reactivated) AS DOUBLE) / COUNT(DISTINCT client_id) AS client_reactivation_rate
FROM client_group
GROUP BY 1
ORDER BY unique_clients_sent DESC
"""

query(apec_split_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,apec_group,unique_clients_sent,reactivated_clients,client_reactivation_rate
0,non-APEC,1161774,815,0.000702
1,APEC,203136,30,0.000148


non-APEC: 1,161,774 clients, 815 reactivated (0.07%) | APEC: 203,136 clients, 30 reactivated (0.01%) — April 2026 alone already shows the same non-APEC > APEC direction the full-window analysis reports, though the gap is far smaller here than it would be if transactional email weren't excluded — see personalization_impact_v2.ipynb's category table for why that exclusion matters.

## Part C — `category_query` (Part 2 of `personalization_impact_v2.ipynb`)

Same shape as Part B, with two differences: `categorized_sends` uses the full eight-way `CASE` instead of the binary one, and transactional/account-service email is kept as its own labeled category here rather than excluded from the population — Part 2 is a descriptive breakdown of every send type, so it's shown rather than hidden (see `personalization_impact_v2.ipynb`'s category table for why its rate isn't a real marketing lever).

### C1 — the `categorized_sends` CTE alone (all categories)

`APEC`, `TRANSACTIONAL`, `DSE`, `TFY`, `dormantbrowseabandon`, `newforyou`, `freshpicks`, else `OTHER` — matched in that order, so a send only falls into a later category if it didn't match an earlier one.

In [12]:
query(f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_campaign
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
)
SELECT
    CASE
        WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC'
        WHEN send_utm_campaign LIKE '%transactional%' THEN 'TRANSACTIONAL'
        WHEN send_utm_campaign LIKE '%dse%' THEN 'DSE'
        WHEN send_utm_campaign LIKE '%tfy%' THEN 'TFY'
        WHEN send_utm_campaign LIKE '%dormantbrowseabandon%' THEN 'dormantbrowseabandon'
        WHEN send_utm_campaign LIKE '%newforyou%' THEN 'newforyou'
        WHEN send_utm_campaign LIKE '%freshpicks%' THEN 'freshpicks'
        ELSE 'OTHER'
    END AS campaign_category,
    count(*) as n_sends, count(distinct client_id) as n_clients
FROM dormant_3yr_sends
GROUP BY 1
ORDER BY n_clients DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,campaign_category,n_sends,n_clients
0,OTHER,22060389,1137020
1,DSE,14287605,958176
2,APEC,1534243,203136
3,TFY,83295,49106
4,TRANSACTIONAL,168924,35352
5,dormantbrowseabandon,9549,8322


freshpicks and newforyou don't show up here at all — both are single one-time sends (July 2025 and January 2026 respectively), so a month that isn't theirs has zero rows for them. Expected, not a bug — see personalization_impact_v2.ipynb's category table for those two categories.

### C2 — the full `category_query`

Same `matched_sessions` → `attributed_demand` → collapse → aggregate structure as Part B, grouped by `campaign_category` instead of `apec_group`. This is the exact query behind Part 2's category table.

In [13]:
category_query = f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_campaign, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE '{MONTH_START}' AND execution_date < DATE '{MONTH_END}'
      AND holdout_group = 0
),
dormant_3yr_sends AS (
    SELECT s.client_id, s.sent_timestamp, s.send_utm_campaign, s.send_utm_content
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
    WHERE j.client_state_detail = 'Dormant'
      AND date_diff('day', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) > {DORMANT_3YR_MIN_DAYS}
),
categorized_sends AS (
    SELECT
        client_id, sent_timestamp, send_utm_content,
        CASE
            WHEN send_utm_campaign LIKE '%apec%' THEN 'APEC'
            WHEN send_utm_campaign LIKE '%transactional%' THEN 'TRANSACTIONAL'
            WHEN send_utm_campaign LIKE '%dse%' THEN 'DSE'
            WHEN send_utm_campaign LIKE '%tfy%' THEN 'TFY'
            WHEN send_utm_campaign LIKE '%dormantbrowseabandon%' THEN 'dormantbrowseabandon'
            WHEN send_utm_campaign LIKE '%newforyou%' THEN 'newforyou'
            WHEN send_utm_campaign LIKE '%freshpicks%' THEN 'freshpicks'
            ELSE 'OTHER'
        END AS campaign_category
    FROM dormant_3yr_sends
),
matched_sessions AS (
    SELECT c.client_id, c.sent_timestamp, c.campaign_category, u.active_session_id
    FROM categorized_sends c
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = c.client_id
        AND u.utm_source = 'blueshift'
        AND u.utm_content = c.send_utm_content
        AND u.datetime_in_utc >= c.sent_timestamp
        AND u.datetime_in_utc <  c.sent_timestamp + INTERVAL '{SESSION_ATTRIBUTION_WINDOW_DAYS}' DAY
    WHERE u.date_in_utc >= DATE '{MONTH_START}'
),
attributed_demand AS (
    SELECT DISTINCT ms.client_id, ms.sent_timestamp, ms.campaign_category
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events e
        ON e.active_session_id = ms.active_session_id
    WHERE e.demand_type IN ('fix', 'direct_buy')
),
client_category AS (
    SELECT
        d.client_id, d.campaign_category,
        MAX(CASE WHEN a.client_id IS NOT NULL THEN 1 ELSE 0 END) AS client_reactivated
    FROM categorized_sends d
    LEFT JOIN attributed_demand a
        ON d.client_id = a.client_id AND d.sent_timestamp = a.sent_timestamp AND d.campaign_category = a.campaign_category
    GROUP BY 1, 2
)
SELECT
    campaign_category,
    COUNT(DISTINCT client_id) AS unique_clients_sent,
    SUM(client_reactivated) AS reactivated_clients,
    CAST(SUM(client_reactivated) AS DOUBLE) / COUNT(DISTINCT client_id) AS client_reactivation_rate
FROM client_category
GROUP BY 1
ORDER BY unique_clients_sent DESC
"""

query(category_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,campaign_category,unique_clients_sent,reactivated_clients,client_reactivation_rate
0,OTHER,1137020,571,0.000502
1,DSE,958176,241,0.000252
2,APEC,203136,30,0.000148
3,TFY,49106,6,0.000122
4,TRANSACTIONAL,35352,2674,0.075639
5,dormantbrowseabandon,8322,1,0.000120


TRANSACTIONAL (7.56%) is far above every other category — see personalization_impact_v2.ipynb's category table for why that's a measurement artifact (a client only gets a transactional send because they're already returning on their own) rather than a real marketing effect. Among the actual marketing categories: OTHER (0.05%) > DSE (0.03%) > APEC (0.01%) > TFY (0.01%) > dormantbrowseabandon (0.01%) — same ranking as the full-window Part 2 table, just on one month's worth of data.